# Extract audio features from EnviSounds dataset
Extracts acoustic features relevant for environmental (non-musical) sounds using librosa.
Writes one JSON per WAV into `data/features/`.

In [34]:
# Install dependencies
%pip install librosa numpy scipy soundfile --quiet

Note: you may need to restart the kernel to use updated packages.


In [36]:
import os
import json
import numpy as np
import librosa
import warnings
warnings.filterwarnings('ignore')

print(f"librosa {librosa.__version__}")

librosa 0.11.0


In [38]:
# Configuration

AUDIO_DIR   = "audio_files"   # folder with category subfolders containing WAVs
OUTPUT_DIR  = "data/features"

SR          = None   # if specified, resamples all files to this rate (None = keep original)
N_FFT       = 2048
HOP_LENGTH  = 1024
N_MFCC      = 13
N_MELS      = 128
MAX_FREQ_HZ = 8000    # crop spectrogram / spectrum display to this frequency
MAX_WAVEFORM_POINTS = 5000

## Features extracted

For more details about the functions used to extract these features see: https://librosa.org/doc/main/index.html 
### Time-domain
| Feature | Description | Why relevant for EnviSounds |
|---|---|---|
| **Waveform** | Raw amplitude signal (downsampled) | Visual inspection |
| **Amplitude envelope** | Mean absolute amplitude per frame | Overview of attack/decay/sustain shape |
| **RMS energy** | Root mean square energy per frame | Loudness over time |
| **Zero crossing rate** | Sign changes per frame | Key feature to distinguish percussive sounds |
| **Temporal centroid** | When in time most energy occurs (single value) | Characterises impulsive vs sustained sounds |
| **Perceptual loudness** | A-weighted RMS — weighted to human hearing sensitivity | More ecologically valid loudness measure |

### Spectral (frame-wise)
| Feature | Description | Why relevant for EnviSounds |
|---|---|---|
| **Spectrogram** | STFT magnitude in dB | Core visual representation |
| **Mel spectrogram** | Spectrogram on perceptual mel scale | Perceptually meaningful frequency axis |
| **Mean power spectrum** | Average spectrum across all frames | Overall frequency profile |
| **Spectral centroid** | Centre of gravity of the spectral energy (i.e. its dominant frequency) | Good indicator of brightness |
| **Spectral rolloff** | Frequency below which 85% of energy falls | Upper frequency extent |
| **Spectral bandwidth** | Spread of energy around centroid | Broad (noise) vs narrow (tonal) |
| **Spectral flatness** | Uniformity of spectral distribution | Key for separating textures (rain, wind) from tonal sources |
| **Spectral contrast** | Energy across sound subbands | Captures harmonic structure |
| **Spectral flux** | Frame-to-frame spectral change | Distinguishes steady textures from dynamic/transient sounds |
| **Spectral kurtosis** | Flatness of the spectral distribution around its mean | High for impulsive sounds (snaps, clicks), low for noise |
| **Log-mel band energies** | Energy across bands | Interpretable frequency-band summary |

### Cepstral
| Feature | Description | Why relevant for EnviSounds |
|---|---|---|
| **MFCCs** | Mel-frequency cepstral coefficients | Most widely used timbre descriptor; works well for envi sound classification |
| **MFCC deltas** | Frame-to-frame MFCC change | Captures temporal dynamics of texture |

### Perceptual / source quality
| Feature | Description | Why relevant for EnviSounds |
|---|---|---|
| **Harmonic-to-noise ratio (HNR)** | Energy in harmonic partials vs noise floor | Separates tonal sources (birdsong, bells) from noise-like textures (wind, rain) |


In [41]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def downsample_array(arr, max_points):
    if len(arr) <= max_points:
        return arr
    idx = np.round(np.linspace(0, len(arr) - 1, max_points)).astype(int)
    return arr[idx]

def to_list(arr, decimals=6):
    return np.round(np.asarray(arr, dtype=float), decimals).tolist()

def a_weighting_rms(y, sr, frame_length, hop_length):
    """A-weighted RMS energy per frame."""
    from scipy.signal import bilinear, lfilter
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    B = [(2 * np.pi * f4) ** 2 * 10 ** (1.99520 / 20), 0, 0, 0, 0]
    A = np.polymul([1, 4 * np.pi * f4, (2 * np.pi * f4) ** 2],
                   [1, 4 * np.pi * f1, (2 * np.pi * f1) ** 2])
    A = np.polymul(np.polymul(A, [1, 2 * np.pi * f3]),
                                  [1, 2 * np.pi * f2])
    b, a = bilinear(B, A, fs=sr)
    y_weighted = lfilter(b, a, y)
    return librosa.feature.rms(y=y_weighted, frame_length=frame_length, hop_length=hop_length)[0]

def spectral_flux(S):
    """L2 norm of frame-to-frame spectral difference. Returns (frames-1,)."""
    diff = np.diff(S, axis=1)
    return np.sqrt(np.sum(diff ** 2, axis=0))

def spectral_kurtosis(S):
    """Spectral kurtosis per frame."""
    S_norm = S / (np.sum(S, axis=0, keepdims=True) + 1e-10)
    freqs  = np.arange(S.shape[0], dtype=float)
    mu1 = np.sum(freqs[:, None] * S_norm, axis=0)
    mu2 = np.sum((freqs[:, None] - mu1[None, :]) ** 2 * S_norm, axis=0)
    mu4 = np.sum((freqs[:, None] - mu1[None, :]) ** 4 * S_norm, axis=0)
    return mu4 / (mu2 ** 2 + 1e-10)

def harmonic_to_noise_ratio(y, hop_length):
    """Per-frame HNR via harmonic/percussive separation."""
    harmonic, noise = librosa.effects.hpss(y)
    rms_h = librosa.feature.rms(y=harmonic, hop_length=hop_length)[0]
    rms_n = librosa.feature.rms(y=noise,    hop_length=hop_length)[0]
    return 10 * np.log10((rms_h ** 2) / (rms_n ** 2 + 1e-10))

def log_mel_band_energies(y, sr, n_fft, hop_length, n_bands=8):
    """Energy summed in n_bands broad mel bands. Returns (n_bands, frames)."""
    S_mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_bands)
    return librosa.power_to_db(S_mel, ref=np.max)

In [43]:
# Feature extraction

def extract_features(wav_path, sr=SR):
    y, sr    = librosa.load(wav_path, sr=sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)

    n_frames    = 1 + (len(y) - N_FFT) // HOP_LENGTH
    frame_times = librosa.frames_to_time(np.arange(n_frames), sr=sr, hop_length=HOP_LENGTH)

    # ── Waveform ── 
    wav_ds = downsample_array(y, MAX_WAVEFORM_POINTS)
    wav_t  = np.linspace(0, duration, len(wav_ds))

    # ── STFT ──
    S      = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
    S_db   = librosa.amplitude_to_db(S, ref=np.max)
    freqs  = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    stft_t = librosa.frames_to_time(np.arange(S.shape[1]), sr=sr, hop_length=HOP_LENGTH)

    freq_mask  = freqs <= MAX_FREQ_HZ
    S_db_crop  = S_db[freq_mask, :]
    S_crop     = S[freq_mask, :]
    freqs_crop = freqs[freq_mask]

    # ── Mel spectrogram ── 
    S_mel     = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    S_mel_db  = librosa.power_to_db(S_mel, ref=np.max)
    mel_freqs = librosa.mel_frequencies(n_mels=N_MELS, fmin=0, fmax=sr/2)

    # ── Time-domain ── 
    envelope  = np.array([np.mean(np.abs(y[i:i+N_FFT])) for i in range(0, len(y)-N_FFT, HOP_LENGTH)])
    rms       = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    zcr       = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    temp_cent = float(np.sum(np.arange(len(y)) * np.abs(y)) / (np.sum(np.abs(y)) + 1e-10) / sr)
    loudness  = a_weighting_rms(y, sr, N_FFT, HOP_LENGTH)

    # ── Spectral ── 
    centroid  = librosa.feature.spectral_centroid( y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)[0]
    rolloff   = librosa.feature.spectral_rolloff(  y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, roll_percent=0.85)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)[0]
    flatness  = librosa.feature.spectral_flatness( y=y,        n_fft=N_FFT, hop_length=HOP_LENGTH)[0]
    contrast  = librosa.feature.spectral_contrast( y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)  # (7, frames)
    flux      = spectral_flux(S_crop)       # (frames-1,)
    kurtosis  = spectral_kurtosis(S_crop)   # (frames,)
    mel_bands = log_mel_band_energies(y, sr, N_FFT, HOP_LENGTH, n_bands=8)  # (8, frames)

    # ── MFCCs ──
    mfcc       = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mfcc_delta = librosa.feature.delta(mfcc)

    # ── HNR ──
    hnr = harmonic_to_noise_ratio(y, HOP_LENGTH)

    # Align to shortest length
    n = min(
        len(frame_times), len(envelope), len(rms), len(zcr), len(loudness),
        len(centroid), len(rolloff), len(bandwidth), len(flatness),
        contrast.shape[1], len(kurtosis), mel_bands.shape[1],
        mfcc.shape[1], S_db_crop.shape[1], S_mel_db.shape[1], len(hnr)
    )
    n_flux = min(n, len(flux))
    ft = frame_times[:n]

    return {
        "info": {
            "duration":          round(float(duration), 4),
            "sample_rate":       int(sr),
            "n_samples":         int(len(y)),
            "temporal_centroid": round(temp_cent, 4),
        },
        "waveform":        {"times": to_list(wav_t, 5),       "samples":       to_list(wav_ds, 6)},
        "envelope":        {"times": to_list(ft, 5),          "values":        to_list(envelope[:n], 6)},
        "rms":             {"times": to_list(ft, 5),          "values":        to_list(rms[:n], 6)},
        "zcr":             {"times": to_list(ft, 5),          "values":        to_list(zcr[:n], 6)},
        "loudness":        {"times": to_list(ft, 5),          "values":        to_list(loudness[:n], 6)},
        "spectrogram":     {"times": to_list(stft_t[:n], 5),  "frequencies":   to_list(freqs_crop, 2), "matrix": to_list(S_db_crop[:, :n], 3)},
        "mel_spectrogram": {"times": to_list(stft_t[:n], 5),  "frequencies":   to_list(mel_freqs, 2),  "matrix": to_list(S_mel_db[:, :n], 3)},
        "spectrum":        {"frequencies": to_list(freqs_crop, 2), "magnitudes": to_list(np.mean(S_db_crop[:, :n], axis=1), 3)},
        "centroid":        {"times": to_list(ft, 5),          "values":        to_list(centroid[:n], 2)},
        "rolloff":         {"times": to_list(ft, 5),          "values":        to_list(rolloff[:n], 2)},
        "bandwidth":       {"times": to_list(ft, 5),          "values":        to_list(bandwidth[:n], 2)},
        "flatness":        {"times": to_list(ft, 5),          "values":        to_list(flatness[:n], 6)},
        "contrast":        {"times": to_list(ft, 5),          "bands":         to_list(contrast[:, :n], 3)},
        "flux":            {"times": to_list(ft[:n_flux], 5), "values":        to_list(flux[:n_flux], 4)},
        "kurtosis":        {"times": to_list(ft, 5),          "values":        to_list(kurtosis[:n], 4)},
        "mel_bands":       {"times": to_list(ft, 5),          "bands":         to_list(mel_bands[:, :n], 3)},
        "mfcc":            {"times": to_list(ft, 5),          "coefficients":  to_list(mfcc[:, :n], 3)},
        "mfcc_delta":      {"times": to_list(ft, 5),          "coefficients":  to_list(mfcc_delta[:, :n], 3)},
        "hnr":             {"times": to_list(ft, 5),          "values":        to_list(hnr[:n], 3)},
    }

In [45]:
# Run extraction

os.makedirs(OUTPUT_DIR, exist_ok=True)

total  = 0
errors = []

for category in sorted(os.listdir(AUDIO_DIR)):
    cat_path = os.path.join(AUDIO_DIR, category)
    if not os.path.isdir(cat_path):
        continue

    out_cat_dir = os.path.join(OUTPUT_DIR, category)
    os.makedirs(out_cat_dir, exist_ok=True)

    wav_files = sorted(f for f in os.listdir(cat_path) if f.lower().endswith('.wav'))

    for wav_file in wav_files:
        wav_path = os.path.join(cat_path, wav_file)
        out_path = os.path.join(out_cat_dir, wav_file.replace('.wav', '.json').replace('.WAV', '.json'))

        try:
            feat = extract_features(wav_path)
            with open(out_path, 'w') as f:
                json.dump(feat, f, separators=(',', ':'))
            print(f"  ✓  {category}/{wav_file}")
            total += 1
        except Exception as e:
            print(f"  ✗  {category}/{wav_file}  —  {e}")
            errors.append((wav_path, str(e)))

print(f"\nDone. {total} files exported to {OUTPUT_DIR}/")
if errors:
    print(f"\n{len(errors)} error(s):")
    for path, msg in errors:
        print(f"  {path}: {msg}")

  ✓  animals/BirdSong_01_MONO.wav
  ✓  animals/BirdSong_02_MONO.wav
  ✓  animals/BirdSong_03_MONO.wav
  ✓  animals/BirdSong_04_MONO.wav
  ✓  animals/BirdSong_05_MONO.wav
  ✓  animals/BirdSong_06_MONO.wav
  ✓  animals/BirdSong_07_MONO.wav
  ✓  animals/BirdSong_08_MONO.wav
  ✓  animals/BirdSong_09_MONO.wav
  ✓  animals/BirdSong_10_MONO.wav
  ✓  animals/BirdSquawk_01_MONO.wav
  ✓  animals/BirdSquawk_02_MONO.wav
  ✓  animals/BirdSquawk_03_MONO.wav
  ✓  animals/BirdSquawk_04new_MONO.wav
  ✓  animals/BirdSquawk_05_MONO.wav
  ✓  animals/BirdSquawk_06_MONO.wav
  ✓  animals/BirdSquawk_07new_MONO.wav
  ✓  animals/BirdSquawk_08_MONO.wav
  ✓  animals/BirdSquawk_09_MONO.wav
  ✓  animals/BirdSquawk_10_MONO.wav
  ✓  animals/BirdsFlyingOff_01_MONO.wav
  ✓  animals/BirdsFlyingOff_02_MONO.wav
  ✓  animals/BirdsFlyingOff_03_MONO.wav
  ✓  animals/BirdsFlyingOff_04_MONO.wav
  ✓  animals/BirdsFlyingOff_05_MONO.wav
  ✓  animals/BirdsFlyingOff_06_MONO.wav
  ✓  animals/BirdsFlyingOff_07_MONO.wav
  ✓  animals/B

In [46]:
# Check

import glob

sample_files = glob.glob(os.path.join(OUTPUT_DIR, '*', '*.json'))
if sample_files:
    with open(sample_files[0]) as f:
        sample = json.load(f)
    print(f"File: {sample_files[0]}")
    print(f"Keys: {list(sample.keys())}")
    print(f"Info: {sample['info']}")
    print(f"Spectrogram:  {len(sample['spectrogram']['frequencies'])} freq bins x {len(sample['spectrogram']['times'])} frames")
    print(f"MFCCs:        {len(sample['mfcc']['coefficients'])} coefficients x {len(sample['mfcc']['times'])} frames")
    print(f"Contrast:     {len(sample['contrast']['bands'])} bands x {len(sample['contrast']['times'])} frames")
    print(f"Mel bands:    {len(sample['mel_bands']['bands'])} bands x {len(sample['mel_bands']['times'])} frames")
else:
    print("No output files found — check AUDIO_DIR path.")

File: data/features/tools/Drilling_08_MONO.json
Keys: ['info', 'waveform', 'envelope', 'rms', 'zcr', 'loudness', 'spectrogram', 'mel_spectrogram', 'spectrum', 'centroid', 'rolloff', 'bandwidth', 'flatness', 'contrast', 'flux', 'kurtosis', 'mel_bands', 'mfcc', 'mfcc_delta', 'hnr']
Info: {'duration': 3.0, 'sample_rate': 44100, 'n_samples': 132300, 'temporal_centroid': 1.5136}
Spectrogram:  372 freq bins x 128 frames
MFCCs:        13 coefficients x 128 frames
Contrast:     7 bands x 128 frames
Mel bands:    8 bands x 128 frames


## Compute per-sound summary statistics

Reads every JSON in data/features/ and computes descriptive statistics (mean, min, max, median, std, range, skewness, IQR) for every feature
<ul>
<li>Scalar frame-wise features: one set of stats per feature</li>
<li>Multi-band / multi-coeff: one set of stats per band/coefficient</li>
    <ul>
<li>contrast: 7 bands -> contrast_0_mean ... contrast_6_iqr</li>
<li>mel_bands: 8 bands -> mel_band_0_mean ... mel_band_7_iqr</li>
<li>mfcc: 13 coeffs -> mfcc_0_mean ... mfcc_12_iqr</li>
<li>mfcc_delta: 13 coeffs -> mfcc_delta_0_mean ... mfcc_delta_12_iqr</li></ul>
</ul>
Output: data/features_stats.csv (one row per sound file, columns = category | sound_name | feature ... )

In [1]:
# Import dependencies and configuration 

import os, json, glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import skew

FEATURES_DIR = "data/features"
CSV_OUT      = "data/features_stats.csv"

# Scalar frame-wise feature keys (module-level so the summary print can use them)
SCALAR_KEYS = [
    ("envelope",  "values"),
    ("rms",       "values"),
    ("zcr",       "values"),
    ("loudness",  "values"),
    ("centroid",  "values"),
    ("rolloff",   "values"),
    ("bandwidth", "values"),
    ("flatness",  "values"),
    ("flux",      "values"),
    ("kurtosis",  "values"),
    ("hnr",       "values"),
]


def _safe_skew(a):
    """Skewness, returning 0.0 for constant arrays (avoids RuntimeWarning)."""
    if a.size < 2 or np.all(a == a[0]):
        return 0.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return float(skew(a))


def _stats(arr, prefix):
    """Return dict of descriptive stats for a 1-D array-like."""
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]          # drop NaN/Inf (e.g. silent HNR frames)
    if a.size == 0:
        keys = ["mean", "min", "max", "median", "std", "range", "skewness", "iqr"]
        return {f"{prefix}_{k}": np.nan for k in keys}
    q25, q75 = np.percentile(a, [25, 75])
    return {
        f"{prefix}_mean":     float(np.mean(a)),
        f"{prefix}_min":      float(np.min(a)),
        f"{prefix}_max":      float(np.max(a)),
        f"{prefix}_median":   float(np.median(a)),
        f"{prefix}_std":      float(np.std(a, ddof=0)),
        f"{prefix}_range":    float(np.max(a) - np.min(a)),
        f"{prefix}_skewness": _safe_skew(a),
        f"{prefix}_iqr":      float(q75 - q25),
    }


def summarise_json(feat):
    """Turn one feature JSON dict into a flat dict of summary statistics."""
    row = {}

    # ── Scalar info fields ──
    row["duration_s"]        = feat["info"]["duration"]
    row["sample_rate"]       = feat["info"]["sample_rate"]
    row["temporal_centroid"] = feat["info"]["temporal_centroid"]

    # ── Scalar frame-wise features ──
    for feat_key, val_key in SCALAR_KEYS:
        row.update(_stats(feat[feat_key][val_key], feat_key))

    # ── Multi-band / multi-coefficient features ──
    # contrast: shape (7, frames) stored as list-of-lists (bands × frames)
    for i, band in enumerate(feat["contrast"]["bands"]):
        row.update(_stats(band, f"contrast_{i}"))

    # mel_bands: shape (8, frames)
    for i, band in enumerate(feat["mel_bands"]["bands"]):
        row.update(_stats(band, f"mel_band_{i}"))

    # mfcc: shape (13, frames)
    for i, coeff in enumerate(feat["mfcc"]["coefficients"]):
        row.update(_stats(coeff, f"mfcc_{i}"))

    # mfcc_delta: shape (13, frames)
    for i, coeff in enumerate(feat["mfcc_delta"]["coefficients"]):
        row.update(_stats(coeff, f"mfcc_delta_{i}"))

    return row


# Main loop

json_files = sorted(glob.glob(os.path.join(FEATURES_DIR, "*", "*.json")))
if not json_files:
    print("No JSON files found in", FEATURES_DIR)
    print("Run the feature extraction cell first.")
else:
    rows = []
    errors = []

    for path in json_files:
        parts      = path.replace("\\", "/").split("/")
        category   = parts[-2]
        sound_name = parts[-1].replace(".json", "")

        try:
            with open(path) as f:
                feat = json.load(f)
            stats = summarise_json(feat)
            stats["category"]   = category
            stats["sound_name"] = sound_name
            rows.append(stats)
        except Exception as e:
            errors.append((path, str(e)))
            print(f"  ✗  {category}/{sound_name}  —  {e}")

    df = pd.DataFrame(rows)

    # Put identifier columns first
    id_cols   = ["category", "sound_name"]
    feat_cols = [c for c in df.columns if c not in id_cols]
    df = df[id_cols + feat_cols]

    os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)
    df.to_csv(CSV_OUT, index=False)

    n_ok  = len(rows)
    n_err = len(errors)
    print(f"Saved {n_ok} rows × {len(df.columns)} columns → {CSV_OUT}")
    if n_err:
        print(f"{n_err} error(s) skipped.")
    print(f"\nColumn groups:")
    print(f"  identifiers : {id_cols}")
    print(f"  info fields : duration_s, sample_rate, temporal_centroid")
    print(f"  scalar feats: {[k for k, _ in SCALAR_KEYS]}")
    print(f"  multi-band  : contrast (7 bands), mel_band (8 bands), mfcc (13), mfcc_delta (13)")
    print(f"  stats per feature/band: mean, min, max, median, std, range, skewness, iqr")
    print(f"\ndf.head():")
    print(df[["category", "sound_name", "duration_s", "rms_mean", "centroid_mean", "mfcc_0_mean"]].head())


Saved 530 rows × 421 columns → data/features_stats.csv

Column groups:
  identifiers : ['category', 'sound_name']
  info fields : duration_s, sample_rate, temporal_centroid
  scalar feats: ['envelope', 'rms', 'zcr', 'loudness', 'centroid', 'rolloff', 'bandwidth', 'flatness', 'flux', 'kurtosis', 'hnr']
  multi-band  : contrast (7 bands), mel_band (8 bands), mfcc (13), mfcc_delta (13)
  stats per feature/band: mean, min, max, median, std, range, skewness, iqr

df.head():
  category        sound_name  duration_s  rms_mean  centroid_mean  mfcc_0_mean
0  animals  BirdSong_01_MONO      3.0000  0.154664    5976.327969  -463.893617
1  animals  BirdSong_02_MONO      3.0000  0.051099    6431.530156  -432.358312
2  animals  BirdSong_03_MONO      3.0000  0.099025    4676.271641  -433.699891
3  animals  BirdSong_04_MONO      3.0000  0.151061    7531.696719  -455.964039
4  animals  BirdSong_05_MONO      2.9996  0.050733    7143.878281  -418.653578
